### openEO

The openEO python client is used to connect to EODC's openEO API. You need to create an account and authenticate when connecting via openEO python client. For more information, see: https://docs.openeo.cloud/join/free_trial.html 

In [1]:
import openeo
from openeo.processes import mean

conn = openeo.connect("https://openeo-dev.eodc.eu/openeo/1.2.0")
conn = conn.authenticate_oidc()
print(conn)

Authenticated using refresh token.
<Connection to 'https://openeo-dev.eodc.eu/openeo/1.2.0/' with OidcBearerAuth>


As soon as you are connected, you can start exploring collections and processes and select the "SENTINEL5P_DAILY_AUT" collection in your loading process. Set the spatial and temporal extent of interest. 

In [2]:
cube_co = conn.load_collection(
    'SENTINEL5P_DAILY_AUT', 
    spatial_extent = dict(west=15, east=17, south=47, north=49, crs="EPSG:4326"), 
    temporal_extent = ["2019-01-01", "2025-12-31"])

You can then use processes, such as "aggregate_temporal_period" to create a monthly mean. 

In [3]:
cube_co_monthly_mean = cube_co.aggregate_temporal_period(period="month", dimension="time", reducer=mean)

In [4]:
cube_save = cube_co_monthly_mean.save_result(format="zarr")

You then need to save your results and select a data format you want to save the data to. Then you can trigger the processing on the openEO API side by using "create_job" and "start_job". 

In [5]:
job_cube = cube_save.create_job(title="Sentinel5P-CO").start_job()

Preflight process graph validation raised: [404] Load call not available for SENTINEL5P_DAILY_AUT


In [8]:
job_cube

<BatchJob job_id='2849269b-bafa-4c49-a895-15751504d7c4'>

Once the job status says "finished", you can get and download your results. 

In [9]:
results = job_cube.get_results()
metadata = results.get_metadata()
results

<JobResults for job '2849269b-bafa-4c49-a895-15751504d7c4'>

In [10]:
results.download_files()

[PosixPath('/home/vhutter/eodc-examples/demos/S5P/EU800M_E048N012T6_20190131T000000.zarr'),
 PosixPath('/home/vhutter/eodc-examples/demos/S5P/job-results.json')]

We can only download files, not folders, so the result is zipped. To make the file available, we rename it and then unzip it.

In [11]:
import os

os.rename("EU800M_E048N012T6_20190131T000000.zarr", "EU800M_E048N012T6_20190131T000000.zip")

In [12]:
from zipfile import ZipFile

with ZipFile("EU800M_E048N012T6_20190131T000000.zip") as zip:
    zip.extractall("EU800M_E048N012T6_20190131T000000.zarr")

Finally, we use the zarr python package to open and explore the results.

In [15]:
import zarr 

dz = zarr.open_group("EU800M_E048N012T6_20190131T000000.zarr")
dz

<Group file://EU800M_E048N012T6_20190131T000000.zarr>

In [18]:
dz.tree()

/
├── carbonmonoxide_total_column (84, 3, 23, 17) float32
├── carbonmonoxide_total_column_n (84, 3, 23, 17) float32
├── qa_threshold (3,) float64
├── spatial_ref () int64
├── time (84,) int64
├── x (17,) float64
└── y (23,) float64

In [24]:
dz["carbonmonoxide_total_column"].shape

(84, 3, 23, 17)

In [29]:
dz["carbonmonoxide_total_column"][:,0,0,0]

array([0.03273358, 0.03332283, 0.03504914, 0.03528113, 0.03090437,
       0.02945343, 0.02854   , 0.03078098, 0.02906286, 0.02810927,
       0.03016327, 0.03106647, 0.03275366, 0.03360337, 0.03569574,
       0.03626114, 0.03329661, 0.02885716, 0.02631419, 0.02770204,
       0.03034489, 0.0320695 , 0.03076357, 0.03240801, 0.0354315 ,
       0.03499338, 0.03597451, 0.03549983, 0.03164957, 0.02935887,
       0.03319725, 0.03629706, 0.03468633, 0.0322985 , 0.03122119,
       0.03080332, 0.03293155, 0.03286538, 0.03453234, 0.0331095 ,
       0.03030166, 0.02704043, 0.02742101, 0.02814465, 0.02799499,
       0.02610219, 0.02707563, 0.02996193, 0.03011239, 0.03131092,
       0.03230166, 0.03187292, 0.03247908, 0.03164896, 0.03487097,
       0.03261745, 0.032011  , 0.0314852 , 0.03167235, 0.03315022,
       0.03335348, 0.03342367, 0.03478356, 0.0328739 , 0.03175216,
       0.02969897, 0.03174963, 0.03451683, 0.03462533, 0.03105334,
       0.03123786, 0.03195443, 0.03132171, 0.03394622, 0.03436

In [30]:
dz["carbonmonoxide_total_column"][:,1,0,0]

array([0.0330771 , 0.03297916, 0.03536473, 0.03506882, 0.03080507,
       0.02955866, 0.02857937, 0.03074181, 0.02905134, 0.02813809,
       0.0305101 , 0.031279  , 0.03270826, 0.03361906, 0.03660991,
       0.03652868, 0.03319067, 0.02923585, 0.02631805, 0.02791556,
       0.03028471, 0.03231764, 0.03054316, 0.03280744, 0.03592758,
       0.03545092, 0.03591679, 0.03545387, 0.03225338, 0.02931103,
       0.03314887, 0.03658452, 0.03484799, 0.03248063, 0.03140974,
       0.03065499, 0.03333255, 0.03295516, 0.03454165, 0.03337736,
       0.03048339, 0.02726816, 0.02734791, 0.02808709, 0.02826647,
       0.02568617, 0.02711783, 0.03017278, 0.03031899, 0.03078679,
       0.03255469, 0.03202037, 0.03308771, 0.03216478, 0.03506357,
       0.03326121, 0.03191733, 0.03069141, 0.03261533, 0.03311396,
       0.03312066, 0.03390155, 0.03447044, 0.03299639, 0.03173469,
       0.02972814, 0.03167353, 0.03449393, 0.03401826, 0.03128133,
       0.03107311, 0.03178624, 0.03115496, 0.03413233, 0.03402